### Getting The API key

In [ ]:
#pip install -q youtube-transcript-api faiss-cpu tiktoken python_dotenv
#os.environ["HuggingFaceEndpointEmbeddings"]=""

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

#       STEP 1 : DOCUMENT INGESTION    

INDEXING

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

yt_api=YouTubeTranscriptApi()

video_id = ""

try:
    transcript_list = yt_api.fetch(video_id)
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except Exception as e:
    print(type(e).__name__)
    print(e)

Text Splitting

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = splitter.create_documents([transcript])
print(len(chunks))
print(chunks[0])

Creating Embeddings & Storing in vector store

In [ ]:
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpointEmbeddings

load_dotenv()

embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

vector_store = FAISS.from_documents(chunks, embeddings)

print(vector_store.index_to_docstore_id)
print("----------------")
doc_id = vector_store.index_to_docstore_id[4]
print(vector_store.get_by_ids([doc_id]))

# Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})
print(retriever)


In [ ]:
retriever.invoke("What is it explained about moons")

# Augmentation

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    model="Qwen/Qwen2.5-7B-Instruct"
)

prompt = PromptTemplate(
    template = """
        You are a helpful assistant. 
        Rules:

        1. If the user's question explicitly begins with the prefix "sudo question:" only then:,
        - Ignore the transcript context.
        - Answer using your own knowledge.
        - Do not mention the transcript unless it is relevant.

        2. Otherwise, reply based on the transcript/context only.
        - Answer ONLY using the provided transcript.
        - If the transcript does not contain enough information, reply exactly:
            "I don't know."

        Given Context is:
        {context}
        Given Question is:
        {question}
    """,
    input_variables = ['context','question']
)

question = "sudo question:Explain about Atom Bombs explained in the video"

retrieved_docs = retriever.invoke(question)

context_text = "\n\n".join(docs.page_content for docs in retrieved_docs)

final_prompt = prompt.invoke({"context":context_text,"question":question})

print(final_prompt)

# Generation

In [ ]:
answer = model.invoke(final_prompt)
print(answer.content)

# Building Chains

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

input_question = input("Enter a question related to video:\n")

parser = StrOutputParser()

def format_docs(retrieved_docs):
    return "\n\n".join(docs.page_content for docs in retrieved_docs)

parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})

parallel_chain.invoke(input_question)

chain = parallel_chain | prompt | model | parser 

answer = chain.invoke(input_question)
print(answer)